<a href="https://colab.research.google.com/github/thanosleggis/Thesis/blob/main/notebooks/EGG_CNN_DEAP_LIME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- 0. ΕΓΚΑΤΑΣΤΑΣΗ ΑΠΑΡΑΙΤΗΤΩΝ ΒΙΒΛΙΟΘΗΚΩΝ ---
!pip install lime

# --- 1. ΕΙΣΑΓΩΓΗ ΒΙΒΛΙΟΘΗΚΩΝ ---
import os
import pickle
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from google.colab import drive
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt

# --- 2. ΣΥΝΔΕΣΗ ΜΕ GOOGLE DRIVE ---
print(" Συνδέομαι με το Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- 3. ΕΥΡΕΣΗ ΦΑΚΕΛΟΥ ΔΕΔΟΜΕΝΩΝ ---
possible_paths = [
    '/content/drive/MyDrive/DEAP.mini',
    '/content/drive/MyDrive/DEAP_mini',
    '/content/drive/MyDrive/deap_mini'
]

DATA_PATH = None
for path in possible_paths:
    if os.path.exists(path):
        files_inside = os.listdir(path)
        dat_count = sum(1 for f in files_inside if f.endswith('.dat'))
        if dat_count > 0:
            DATA_PATH = path
            print(f"\n Βρέθηκε ο φάκελος: {DATA_PATH}")
            print(f" Περιέχει {dat_count} αρχεία .dat")
            break

if DATA_PATH is None:
    print(" Δεν βρέθηκε ο φάκελος")
else:
    # --- 4. ΡΥΘΜΙΣΕΙΣ (HYPERPARAMETERS) ---
    WINDOW_SIZE = 128
    STRIDE = 64
    CHANNELS = 32
    EPOCHS = 2  # Αν θέλεις να τρέξει γρήγορα, κάντο 2-3
    BATCH_SIZE = 128

    # --- 5. ΣΥΝΑΡΤΗΣΗ ΦΟΡΤΩΣΗΣ ΔΕΔΟΜΕΝΩΝ ---
    def load_all_subjects(folder):
        files = [f for f in os.listdir(folder) if f.endswith('.dat')]
        files.sort()

        all_X = []
        all_y = []


        for idx, filename in enumerate(files):
            file_path = os.path.join(folder, filename)

            if idx % 5 == 0:
                print(f"    Φόρτωση {idx+1}/{len(files)}: {filename}...")

            try:
                with open(file_path, 'rb') as f:
                    content = pickle.load(f, encoding='latin1')

                data = content['data']
                labels = content['labels']

                for i in range(len(data)):
                    trial_signal = data[i, :32, :].T.astype('float32')
                    clean_signal = trial_signal[384:, :] # Baseline removal
                    label = 0 if labels[i, 0] < 5 else 1 # Label Valence

                    for start in range(0, clean_signal.shape[0] - WINDOW_SIZE, STRIDE):
                        end = start + WINDOW_SIZE
                        window = clean_signal[start:end, :]
                        if window.shape[0] == WINDOW_SIZE:
                            all_X.append(window)
                            all_y.append(label)

            except Exception as e:
                print(f" Σφάλμα στο {filename}: {e}")

        return np.array(all_X, dtype='float32'), np.array(all_y, dtype='int')

    # --- 6. ΕΚΤΕΛΕΣΗ ΠΡΟΕΤΟΙΜΑΣΙΑΣ ---
    X, y = load_all_subjects(DATA_PATH)

    if len(X) > 0:
        print(f"\n ΟΛΑ τα δεδομένα φορτώθηκαν!")
        print(f" Σύνολο δειγμάτων: {X.shape[0]}")

        y = to_categorical(y, 2)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        print(f" Εκπαίδευση σε {len(X_train)} δείγματα, Test σε {len(X_test)} δείγματα.")

        # --- 7. ΚΑΤΑΣΚΕΥΗ ΜΟΝΤΕΛΟΥ (CNN-RNN) ---
        inputs = Input(shape=(WINDOW_SIZE, CHANNELS))

        x = Conv1D(64, 3, activation='relu', padding='same')(inputs)
        x = BatchNormalization()(x)
        x = MaxPooling1D(2)(x)
        x = Dropout(0.3)(x)

        x = Conv1D(128, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPooling1D(2)(x)
        x = Dropout(0.3)(x)

        x = LSTM(128)(x)
        x = Dropout(0.4)(x)

        x = Dense(64, activation='relu')(x)
        outputs = Dense(2, activation='softmax')(x)

        model = Model(inputs, outputs)
        model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])

        # --- 8. ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ ---
        print("\n Έναρξη Εκπαίδευσης (Full Dataset)...")
        history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_test, y_test))

        # --- 9. ΑΞΙΟΛΟΓΗΣΗ ΜΟΝΤΕΛΟΥ ---
        loss, acc = model.evaluate(X_test, y_test)
        print("\n========================================")
        print(f" ΤΕΛΙΚΗ ΑΚΡΙΒΕΙΑ (Γενικό Μοντέλο): {acc*100:.2f}%")
        print("========================================")

        # --- 10. ΕΠΕΞΗΓΗΣΙΜΟΤΗΤΑ (LIME EXPLAINABLE AI) ---
        print("\n Ξεκινάει η ανάλυση Επεξηγησιμότητας (LIME)...")

        # 10.1 "Ισιώνουμε" (flatten) τα δεδομένα για να τα διαβάσει το LIME
        num_features = WINDOW_SIZE * CHANNELS
        X_train_flat = X_train[:100].reshape(-1, num_features)
        X_test_flat = X_test.reshape(-1, num_features)

        # 10.2 Δημιουργούμε ονόματα για τα χαρακτηριστικά (π.χ. "Ch5_Time120")
        feature_names = [f"Ch{c}_Time{t}" for t in range(WINDOW_SIZE) for c in range(CHANNELS)]

        # 10.3 Ορίζουμε μια συνάρτηση που μετατρέπει τα flat δεδομένα πίσω σε 3D για το μοντέλο
        def predict_fn(x_flat):
            x_reshaped = x_flat.reshape(-1, WINDOW_SIZE, CHANNELS)
            return model.predict(x_reshaped, verbose=0)

        # 10.4 Δημιουργία του LIME Explainer
        explainer = lime.lime_tabular.LimeTabularExplainer(
            training_data=X_train_flat,
            feature_names=feature_names,
            class_names=['Low Valence', 'High Valence'],
            mode='classification'
        )

        sample_idx = 0
        print(f"\n Ανάλυση Δείγματος {sample_idx} με το LIME:")

        # 10.5 Εξήγηση της πρόβλεψης για το δείγμα 0 (ζητάμε τα 15 πιο σημαντικά σημεία)
        exp = explainer.explain_instance(
            data_row=X_test_flat[sample_idx],
            predict_fn=predict_fn,
            num_features=15,
            top_labels=1
        )

        # Εμφάνιση του γραφήματος του LIME
        predicted_class = exp.available_labels()[0]
        fig = exp.as_pyplot_figure(label=predicted_class)
        plt.title(f"LIME: Top 15 Σημεία που επηρέασαν την πρόβλεψη")
        plt.tight_layout()
        plt.show()

        # --- 11. SUBJECT-SPECIFIC FINE-TUNING ---
        print("\n Ξεκινάει το Subject-Specific Fine-Tuning...")

        def load_single_subject(folder, filename):
            file_path = os.path.join(folder, filename)
            with open(file_path, 'rb') as f:
                content = pickle.load(f, encoding='latin1')

            data = content['data']
            labels = content['labels']

            X_sub, y_sub = [], []
            for i in range(len(data)):
                trial_signal = data[i, :32, :].T.astype('float32')
                clean_signal = trial_signal[384:, :]
                label = 0 if labels[i, 0] < 5 else 1

                for start in range(0, clean_signal.shape[0] - WINDOW_SIZE, STRIDE):
                    end = start + WINDOW_SIZE
                    window = clean_signal[start:end, :]
                    if window.shape[0] == WINDOW_SIZE:
                        X_sub.append(window)
                        y_sub.append(label)

            return np.array(X_sub, dtype='float32'), to_categorical(np.array(y_sub, dtype='int'), 2)

        subject_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.dat')]
        target_subject = random.choice(subject_files)

        print(f" Επιλέχθηκε τυχαία ο εθελοντής: {target_subject}")
        print(f" Φόρτωση δεδομένων για τον εθελοντή: {target_subject}")
        X_sub, y_sub = load_single_subject(DATA_PATH, target_subject)

        X_sub_train, X_sub_test, y_sub_train, y_sub_test = train_test_split(X_sub, y_sub, test_size=0.2, random_state=42)

        loss_before, acc_before = model.evaluate(X_sub_test, y_sub_test, verbose=0)
        print(f" Ακρίβεια στον εθελοντή ΠΡΙΝ το Fine-Tuning: {acc_before*100:.2f}%")

        for layer in model.layers[:-4]:
            layer.trainable = False

        model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

        print("\n Ξεκινάει η μικρο-ρύθμιση (Fine-Tuning) για 10 epochs...")
        history_ft = model.fit(X_sub_train, y_sub_train, epochs=10, batch_size=32, validation_data=(X_sub_test, y_sub_test))

        loss_after, acc_after = model.evaluate(X_sub_test, y_sub_test, verbose=0)
        print("\n========================================")
        print(f" ΑΚΡΙΒΕΙΑ ΣΤΟΝ ΕΘΕΛΟΝΤΗ '{target_subject}' ΜΕΤΑ ΤΟ FINE-TUNING: {acc_after*100:.2f}%")

        improvement = (acc_after - acc_before) * 100
        if improvement > 0:
            print(f" Συνολική βελτίωση: +{improvement:.2f}% ")
        else:
            print(f" Διαφορά: {improvement:.2f}% (Το μοντέλο είχε ήδη πιάσει την οροφή του!)")
        print("========================================")

    else:
        print(" Πρόβλημα: Δεν βρέθηκαν δεδομένα.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=4683e0ebed6027b291fb18e8dd97d107cb2d23de3c1fe07619e8725d9a6157fb
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
🔌 Συνδέομαι με το Google Drive...
